<p><small><small>This Notebook is made available subject to the licence and terms set out in the <a href = "http://www.github.com/google-deepmind/ai-foundations">AI Research Foundations Github README file</a>.</small></small></p>

<img src="https://storage.googleapis.com/dm-educational/assets/ai_foundations/GDM-Labs-banner-image-C7.png">

# Lab: Data Generation for Conversational Model

<a href='https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_8/gdm_lab_8_7_conversation_data_generation.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg' alt='Open In Colab'/></a>

1 hour

**Project pathway:** Conversational

This notebook is a worked example that demonstrates how you can programatically turn a dataset with short paragraphs into a Q&A dataset. This notebook does not contain any specific activities but you may want to adapt it if you want to apply similar transformations to a different dataset.

## Imports

This notebook uses the Pandas package for storing and processing data, as well as standard Python libraries like `random`, `re`, and `collections`.

In [ ]:
import pandas as pd # For loading and processing data.
import random # For random sampling.
import re # For defining regular expressions.
import json # For writing data to a JSONL file.

from google.colab import drive # For mounting Google Drive.

from typing import List, Tuple, Dict, Any
import collections

Counter = collections.Counter

## Load the data

This notebook uses the Africa Galore Dataset and downloads it from a public URL. If you would like to use your own dataset or store the processed data, it is recommended that you connect Colab to Google Drive.

### Connect Colab with Google Drive

The following cell mounts your Google Drive so that you can access files by adding `/content/drive/MyDrive/` to the beginning of your file path. Running this cell will open a dialog that asks you whether you want to connect Colab to Google Drive. Follow the instructions and enable all permissions, so that Colab can read and write files from Google Drive.

In [ ]:
drive.mount('/content/drive')

The following cell downloads the Africa Galore dataset, stores it in a Pandas dataframe, and prints some basic statistics.

In [ ]:
# Download the file from a link.
filepath = "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore_large.json"

# Read with pandas.
df = pd.read_json(filepath, lines=True)

# Other file-type examples:
# For loading JSON files.
# df = pd.read_json(filepath, orient="records")

# For loading a CSV file.
# df = pd.read_csv(filepath, encoding="utf-8")

# If you are using an API, implement a function, such as `load_from_api()`
# that downloads the data using an API.
# df = load_from_api()

# Take a quick look at the first few rows.
display(df.head())

print("Columns:", df.columns.tolist())
print("Number of rows:", len(df))

# Display unique values from the 'category' column.
print("Unique categories:", df["category"].unique())

## Create synthetic Q&A pairs from non-conversational datasets

The following cells implement several helper functions that aim to extract countries and regions from the descriptions in the Africa Galore dataset. Using these helper functions, the code automatically generates prompt and response pairs. All of this is limited to content discussing food (examples having the category "Food").

If you would like to adapt this code for your own project, consider these pointers for getting started with modifying the code:

1. You may change or expand the included topics (line with `TOPICS = ["food"]`)
   ```python
    # Current: Only food topics
    TOPICS = ["food"]
    ```


2. You may add more prompt templates in the `build_prompt()` function.
    ```python
    # Add after the existing food_prompts:
    place_prompts = [
      f"Name a must-see place in {location}?",
      f"What is a top attraction to visit in {location}?",
      f"Suggest a famous landmark from {location}?"
    ]
    ```

3. If you also need to extract geographic regions, you may improve geographic matching (in the `assign_region()` function, around the line  `country.lower() in name_text`).
    ```python
    # Current simple matching:
    if (country.lower() in name_text or country.lower() in desc_text):
    ```

    E.g., better matching techniques using regular expressions:
    ```python
    country_pattern = r'\b' + re.escape(country.lower()) + r'\b'
    if (re.search(country_pattern, name_text) or re.search(country_pattern, desc_text)):
    ```

4. Adjust the size of the dataset (in `generate_dataset()`) or the length of the responses (in `generate_responses()`).



The conversational model in the worked example needs to recognize African regions and identify where countries are located. This helps it provide better responses when users ask about food in broader regions rather than specific countries.

In [ ]:
# Map countries to African regions.
# Geographic regions adopted using the United Nations methodology.
# https://unstats.un.org/unsd/methodology/m49/

COUNTRY_TO_REGION = {
    # Northern Africa.
    "Algeria":                          "Northern Africa",
    "Egypt":                            "Northern Africa",
    "Libya":                            "Northern Africa",
    "Morocco":                          "Northern Africa",
    "Sudan":                            "Northern Africa",
    "Tunisia":                          "Northern Africa",
    "Western Sahara":                   "Northern Africa",
    # Eastern Africa.
    "British Indian Ocean Territory":   "Eastern Africa",
    "Burundi":                          "Eastern Africa",
    "Comoros":                          "Eastern Africa",
    "Djibouti":                         "Eastern Africa",
    "Eritrea":                          "Eastern Africa",
    "Ethiopia":                         "Eastern Africa",
    "French Southern Territories":      "Eastern Africa",
    "Kenya":                            "Eastern Africa",
    "Madagascar":                       "Eastern Africa",
    "Malawi":                           "Eastern Africa",
    "Mauritius":                        "Eastern Africa",
    "Mayotte":                          "Eastern Africa",
    "Mozambique":                       "Eastern Africa",
    "Réunion":                          "Eastern Africa",
    "Rwanda":                           "Eastern Africa",
    "Seychelles":                       "Eastern Africa",
    "Somalia":                          "Eastern Africa",
    "South Sudan":                      "Eastern Africa",
    "Uganda":                           "Eastern Africa",
    "Tanzania":                         "Eastern Africa",
    "Zambia":                           "Eastern Africa",
    "Zimbabwe":                         "Eastern Africa",
    # Middle Africa.
    "Angola":                           "Middle Africa",
    "Cameroon":                         "Middle Africa",
    "Central African Republic":         "Middle Africa",
    "Chad":                             "Middle Africa",
    "Congo":                            "Middle Africa",
    "Democratic Republic of the Congo": "Middle Africa",
    "Equatorial Guinea":                "Middle Africa",
    "Gabon":                            "Middle Africa",
    "Sao Tome and Principe":            "Middle Africa",
    # Southern Africa.
    "Botswana":                         "Southern Africa",
    "Eswatini":                         "Southern Africa",
    "Lesotho":                          "Southern Africa",
    "Namibia":                          "Southern Africa",
    "South Africa":                     "Southern Africa",
    # Western Africa.
    "Benin":                            "Western Africa",
    "Burkina Faso":                     "Western Africa",
    "Cape Verde":                       "Western Africa",
    "Côte d'Ivoire":                    "Western Africa",
    "Gambia":                           "Western Africa",
    "Ghana":                            "Western Africa",
    "Guinea":                           "Western Africa",
    "Guinea-Bissau":                    "Western Africa",
    "Liberia":                          "Western Africa",
    "Mali":                             "Western Africa",
    "Mauritania":                       "Western Africa",
    "Niger":                            "Western Africa",
    "Nigeria":                          "Western Africa",
    "Saint Helena":                     "Western Africa",
    "Senegal":                          "Western Africa",
    "Sierra Leone":                     "Western Africa",
    "Togo":                             "Western Africa"
}

# The example only uses 'food' anyway. You can use this list to
# restrict your Q&A pairs construction to specific topics if your
# processed dataset contains multiple categories/topics.
TOPICS = ["food"]

# Filter dataframe such that it only contains examples with categories in
# TOPICS.
df = df[df["category"].str.lower().isin(TOPICS)]

The functions below convert structured textual information into conversational training examples for  fine-tuning. The `assign_region` function tags dataset entries with their country and UN-defined region (if matched). The `build_prompt` function then builds natural-sounding user prompts about African food. The `generate_response` function selects matching entries and crafts appropriate responses. The `generate_dataset` function assembles all pairs into a balanced dataset. Randomized templates and proportional sampling help increase diversity and regional coverage.

In [ ]:
def assign_region(df: pd.DataFrame) -> pd.DataFrame:
    """Tag each dataset entry with its corresponding country and UN region.

    Args:
      df: Pandas DataFrame containing at least 'name' and 'description' columns.

    Returns:
      DataFrame with two additional columns: 'country' and 'region'.
    """
    df["country"] = "Unknown"
    df["region"] = "Unknown"

    # Process each row individually to ensure proper country-region mapping.
    for idx, row in df.iterrows():
        name_text = str(row.get("name", "")).lower()
        desc_text = str(row.get("description", "")).lower()

        # Find the first matching country and assign both country and region.
        # Simple string-based search. It can be further improved.
        for country, region in COUNTRY_TO_REGION.items():
            if country.lower() in name_text or country.lower() in desc_text:
                df.loc[idx, "country"] = country
                df.loc[idx, "region"] = region
                # Stop at first match to avoid overwriting and simplify the
                # downstream logic and reduce ambiguity. You can try to improve
                # this by handling multi-country/regions assignments.
                break

    return df


def build_prompt(location: str, topic: str) -> str:
    """Create a user prompt for a given topic and location.

    Create a user prompt for a given topic and location, with multiple natural
    language variations.

    Args:
      location: Country or region name.
      topic: Can be expanded, e.g. ['place', 'food', 'climate'].
      n: Number of items to request.

    Returns:
      String containing a randomly chosen user prompt.
    """

    food_prompts = [
        f"List a traditional food to try in {location}.",
        f"What is one popular dish from {location}?",
        f"I'm visiting {location}; what food should I definitely try?",
        f"Recommend a well-known local dish from {location}.",
        f"Tell me about a signature food from {location}.",
    ]

    # Pick a random prompt variant. The example case only has 'food'.
    # You can expand it to support more topics.
    if topic == "food":
        return random.choice(food_prompts)
    # Default fallback prompt.
    return f"Tell me something about {topic} in {location}."


def generate_response(pool: pd.DataFrame, topic: str) -> Tuple[str, pd.Series]:
    """Generate a single valid response.

    Generate a single valid response entry (one example only). This function is
    used to showcase LoRA fine-tuning on a concise, expressive style.

    Returns:
      tuple: (response_text, row_data) for consistent metadata generation
    """
    row = pool.sample(1).iloc[0]
    # Truncate description to avoid cutting off mid-word and fit in memory.
    desc = row["description"][:400].strip()
    # If cut off mid-word, find the last complete word.
    if len(row["description"]) > 400 and not desc.endswith("."):
        last_space = desc.rfind(" ")
        if (
            last_space > 350
        ):  # Only truncate at word boundary if have enough text.
            desc = desc[:last_space]

    country = row.get("country", "").strip()
    region = row.get("region", "").strip()

    # Build location string with both country and region when available.
    location_parts = []
    if country and country.lower() != "unknown":
        location_parts.append(country)
    if region and region.lower() != "unknown" and region != country:
        location_parts.append(region)

    if not location_parts:
        location_text = "Africa"
    elif len(location_parts) == 1:
        location_text = location_parts[0]
    else:
        location_text = f"{location_parts[0]} in {location_parts[1]}"

    # Build response with consistent format. This can be used to
    # demonstrate the efficiency of the fine-tuning later.
    if topic == "food":
        response_text = (
            f"This dish captures the authentic flavor and spirit of "
            f"{location_text}. "
            f"{row['name']} — {desc}"
        )

    return response_text, row


def generate_dataset(
    df: pd.DataFrame, target_size: int = 500
) -> List[Dict[str, Any]]:
    """Generate synthetic multi-turn chat pairs for LoRA fine-tuning.

    Args:
      df: DataFrame with 'category', 'name', and 'description' columns.
      target_size: Total number of Q&A pairs to create.

    Returns:
      List of dicts with 'prompt', 'response', and 'meta' fields.
    """
    df = assign_region(df)
    df = df[df["region"] != "Unknown"].copy()
    data = []

    total_entries = len(df)
    for topic in TOPICS:
        topic_df = df[df["category"].str.lower() == topic]
        if topic_df.empty:
            continue

        # Proportional allocation: this distributes the total target size across
        # different topics based on their relative share in the dataset.
        # Currently, since only one topic ('food') is used, this has no effect,
        # but it enables easy scaling if new topics (e.g. 'place', 'climate')
        # are added later.
        per_topic = int(target_size * (len(topic_df) / total_entries))

        for col in ["region", "country"]:
            for loc in topic_df[col].unique():
                subset = topic_df[topic_df[col] == loc]
                if subset.empty:
                    continue
                for _ in range(
                    max(1, per_topic // (2 * len(topic_df[col].unique())))
                ):
                    prompt = build_prompt(loc, topic)
                    response, used_row = generate_response(subset, topic)

                    data.append(
                        {
                            "prompt": f"{prompt}",
                            "response": response,
                            "meta": {
                                "topic": topic,
                                "region": used_row.get("region", None),
                                "country": used_row.get("country", None),
                            },
                        }
                    )

    # Remove any exact duplicates.
    seen = {}
    for d in data:
        seen[f"{d['prompt']}|{d['response']}"] = d
    return list(seen.values())

After defining all Q&A generation functions, the next step creates the  synthetic dataset for fine-tuning. The call to `generate_dataset(df)` processes the input data, applies regional tagging, builds question-answer pairs, and compiles them into a structured list.

In [ ]:
chat_data = generate_dataset(df)
display(df.head())

A summary report showing how many examples were created and how they are distributed across topics, regions, and
countries is also generated. This helps verify dataset balance and geographic coverage before model training.

In [ ]:
# Generate a distribution report.
print(f"\n{'=' * 50}")
print(f"Generated {len(chat_data)} unique chat-style continuation examples")
print(f"{'=' * 50}")

topic_counts = Counter(d["meta"]["topic"] for d in chat_data)

region_counts = Counter(
    d["meta"]["region"] for d in chat_data if d["meta"]["region"]
)
country_counts = Counter(
    d["meta"]["country"] for d in chat_data if d["meta"]["country"]
)

print(f"\n{'=' * 50}")
print(f"Distribution Report")
print(f"{'=' * 50}")

print("By topic:")
for topic, count in topic_counts.items():
    print(f"  {topic}: {count}")

print("\nBy region:")
for region, count in sorted(region_counts.items()):
    print(f"  {region}: {count}")

print("\nTop 10 countries:")
for country, count in country_counts.most_common(10):
    print(f"  {country}: {count}")

------
> 💡 **Tip: Improve string matching method**
>
> The current `assign_region()` function uses a simple *string-containment* approach; it just checks if a country name appears anywhere in the `name` or `description` text.  
> This works for clear cases, but has several limitations:
> - It can miss countries when names are spelled differently, abbreviated, or indirectly mentioned.  
> - It cannot detect regional references.  
> - Struggles with ambiguous or multilingual text.  
>
> You can make this smarter without adding major infrastructure or deep learning models. For example:
>
> - **Cosine similarity with TF-IDF vectors** using scikit-learn’s `TfidfVectorizer` and `cosine_similarity`.  
>   This lets you match text and country names based on contextual overlap rather than exact words.  
>   See [Scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) example.
>
> - **Fuzzy string matching** with libraries like [FuzzyWuzzy (RapidFuzz)](https://rapidfuzz.github.io/RapidFuzz/) to handle typos and minor differences.  
>   Example: `'Cote d’Ivoire'` and `'Ivory Coast'`.
>
> - **Sentence embeddings** using [Universal Sentence Encoder (USE)](https://www.tensorflow.org/hub/tutorials/semantic_similarity_with_tf_hub_universal_encoder).  
>   Even though it is more advanced, USE works out-of-the-box for semantic similarity (no retraining required) and can recognize related phrases.
>
> These techniques can be combined or tested separately to find the balance between accuracy and computational cost.  
> For smaller datasets, scikit-learn and RapidFuzz are often enough. USE or similar embedding models become useful when the text is long, varied, or context-rich.
>
> The methods listed above are examples, not an exhaustive list. Other similarity-based or NLP-driven techniques may also be suitable depending on the dataset size, language diversity, and performance goals.
>
------

## Save the converted dataset to Google Drive

Run the following cell to save the converted dataset to Google Drive. This will allow you to further process the dataset and use it for fine-tuning a conversational model in later parts of this capstone.

In [ ]:
# Save chat_data to a JSONL file.
with open(
    "/content/drive/MyDrive/africa_galore_qa_food.jsonl", "w", encoding="utf-8"
) as f:
    for d in chat_data:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")

## Summary

This notebook showed you how to automatically turn descriptive paragraphs in the Africa Galore dataset into pairs of prompts and responses that could be used for training a conversational model. If you are also converting a dataset with information into prompt-response pairs, this worked example may be a good starting point for building your conversational dataset.